In [1]:
import pandas as pd
import re
original_df = pd.read_json("../02-raw/k8s_incidents_transformed.jsonl", lines=True)

original_df.head()

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df = original_df[["scenario_id", "pod_describe", "pod_logs", "pod_logs_previous"]].copy()

In [ ]:
df.head()

In [ ]:
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.lower() in {"nan", "none"}:
        return ""
    x = x.replace("\r\n", "\n").replace("\r", "\n").replace("\t", "    ")
    lines = [line.rstrip() for line in x.split("\n")]
    return "\n".join(lines).strip()

In [ ]:
for col in ["pod_describe", "pod_logs", "pod_logs_previous"]:
    df[col] = df[col].apply(normalize_text)

In [ ]:
def extract_first(pattern, text, flags=re.IGNORECASE | re.MULTILINE):
    if not text:
        return None
    m = re.search(pattern, text, flags)
    return m.group(1).strip() if m else None

In [ ]:
def extract_block(text, block_name):
    if not text:
        return None

    lines = text.split("\n")
    block = []
    in_block = False

    for line in lines:
        if not in_block:
            if re.match(rf"^{re.escape(block_name)}:\s*$", line.strip()):
                in_block = True
            continue

        if line and not line.startswith(" ") and re.match(r"^[A-Za-z0-9_.\-/ ]+:\s*", line):
            break

        block.append(line)

    out = "\n".join(block).strip()
    return out if out else None

In [ ]:
def parse_events_block(events_text):
    if not events_text:
        return {
            "event_reason": None,
            "event_message": None,
            "all_event_reasons": [],
            "all_event_messages": []
        }

    reasons = []
    messages = []

    lines = [line for line in events_text.split("\n") if line.strip()]

    for line in lines:
        if "Reason" in line and "Message" in line:
            continue
        if line.strip().startswith("----"):
            continue

        parts = re.split(r"\s{2,}", line.strip())
        if len(parts) >= 5:
            reasons.append(parts[1].strip())
            messages.append(parts[-1].strip())

    return {
        "event_reason": reasons[0] if reasons else None,
        "event_message": messages[0] if messages else None,
        "all_event_reasons": reasons,
        "all_event_messages": messages
    }

In [ ]:
def parse_describe(text):
    containers_block = extract_block(text, "Containers")
    volumes_block = extract_block(text, "Volumes")
    tolerations_block = extract_block(text, "Tolerations")
    events_block = extract_block(text, "Events")

    events = parse_events_block(events_block)

    return {
        "pod_name": extract_first(r"^Name:\s*(.+)$", text),
        "namespace": extract_first(r"^Namespace:\s*(.+)$", text),
        "service_account_name": extract_first(r"^Service Account:\s*(.+)$", text),
        "node": extract_first(r"^Node:\s*(.+)$", text),
        "pod_status": extract_first(r"^Status:\s*(.+)$", text),
        "pod_ip": extract_first(r"^IP:\s*(.+)$", text),
        "controlled_by": extract_first(r"^Controlled By:\s*(.+)$", text),
        "image": extract_first(r"^\s*Image:\s*(.+)$", containers_block or text),
        "container_state": extract_first(r"^\s*State:\s*(.+)$", containers_block or text),
        "last_state": extract_first(r"^\s*Last State:\s*(.+)$", containers_block or text),
        "ready": extract_first(r"^\s*Ready:\s*(.+)$", containers_block or text),
        "restart_count": extract_first(r"^\s*Restart Count:\s*(.+)$", containers_block or text),
        "node_selectors": extract_first(r"^Node-Selectors:\s*(.+)$", text),
        "qos_class": extract_first(r"^QoS Class:\s*(.+)$", text),
        "claim_name": extract_first(r"^\s*ClaimName:\s*(.+)$", volumes_block or ""),
        "describe_events_raw": events_block,
        "describe_containers_raw": containers_block,
        "describe_volumes_raw": volumes_block,
        "describe_tolerations_raw": tolerations_block,
        "event_reason": events["event_reason"],
        "event_message": events["event_message"],
        "all_event_reasons": events["all_event_reasons"],
        "all_event_messages": events["all_event_messages"],
    }

In [ ]:
describe_parsed = df["pod_describe"].apply(parse_describe).apply(pd.Series)
df = pd.concat([df, describe_parsed], axis=1)

In [ ]:
def build_evidence_text(row):
    parts = []

    if row.get("pod_describe"):
        parts.append("POD DESCRIBE:\n" + row["pod_describe"])

    if row.get("pod_logs"):
        parts.append("POD LOGS:\n" + row["pod_logs"])

    if row.get("pod_logs_previous"):
        parts.append("POD LOGS PREVIOUS:\n" + row["pod_logs_previous"])

    return "\n\n".join(parts).strip() if parts else None

In [ ]:
df["evidence_text"] = df.apply(build_evidence_text, axis=1)

In [ ]:
parsed_df = df[
    [
        "scenario_id",
        "namespace",
        "pod_name",
        "service_account_name",
        "node",
        "pod_status",
        "image",
        "container_state",
        "last_state",
        "ready",
        "restart_count",
        "node_selectors",
        "claim_name",
        "event_reason",
        "event_message",
        "evidence_text",
    ]
].copy()

In [ ]:
parsed_df.head(20)

In [ ]:
preview_df = parsed_df.copy()

for col in ["event_message", "evidence_text"]:
    if col in preview_df.columns:
        preview_df[col] = preview_df[col].fillna("").str.slice(0, 200)

preview_df

In [ ]:
final_df = df.copy()

In [ ]:
cols = [
    "scenario_id",
    "namespace",
    "pod_name",
    "service_account_name",
    "node",
    "pod_status",
    "image",
    "container_state",
    "last_state",
    "ready",
    "restart_count",
    "node_selectors",
    "claim_name",
    "event_reason",
    "event_message",
    "pod_describe",
    "pod_logs",
    "pod_logs_previous",
    "evidence_text",
]

In [ ]:
final_df = df[cols].copy()
final_df.head()